# Recreate cell 4 using the saved model
Run the cell below. It loads `nn_model_conv_MPE.pth` and `normalization_conv_MPE.npz` and uses `combined.csv` for the observed curves. Place these three files beside this notebook or in `~/Downloads`, or edit `ASSET_DIR`. Requires NumPy, PyTorch, and Matplotlib.

No training is performed. The original preprocessing, seed-0 test split, test examples 20–24, and plotting code are preserved. Predictions come from the supplied checkpoint; they match the old figure only to the extent that checkpoint and CSV match the originals.


In [ ]:
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, random_split
import matplotlib.pyplot as plt

# Requires numpy, torch, and matplotlib. This cell only loads and evaluates.
# Look beside the notebook first, then in Downloads; edit ASSET_DIR if needed.
ASSET_DIR = Path.home() / "Downloads"

def asset_path(name):
    for directory in (Path.cwd(), ASSET_DIR):
        path = directory / name
        if path.is_file():
            return path
    raise FileNotFoundError(f"Cannot find {name}; set ASSET_DIR to its folder.")

# Same architecture and SiLU activations as the original notebook.
class NeuralNetwork(nn.Module):
    def __init__(self, input_dim=3, output_dim=20, width=75, depth=3):
        super().__init__()
        layers = [nn.Linear(input_dim, width), nn.SiLU()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.SiLU()]
        layers.append(nn.Linear(width, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# Use CPU so the notebook also runs on machines without a GPU.
state_dict = torch.load(
    asset_path("nn_model_conv_MPE.pth"), map_location="cpu", weights_only=True
)
model = NeuralNetwork()
model.load_state_dict(state_dict, strict=True)
model.eval()
model.requires_grad_(False)

# Use the saved normalization parameters; do not recompute them from the data.
with np.load(asset_path("normalization_conv_MPE.npz"), allow_pickle=False) as norms:
    X_mu = torch.tensor(norms["X_mu"], dtype=torch.float32)
    X_sigma = torch.tensor(norms["X_sigma"], dtype=torch.float32)
    Y_mu = torch.tensor(norms["Y_mu"], dtype=torch.float32)
    Y_sigma = torch.tensor(norms["Y_sigma"], dtype=torch.float32)

def normX(X):
    return (X - X_mu) / X_sigma

def denormy(Y):
    return Y * Y_sigma + Y_mu

# The observed data curves require combined.csv, as in the original notebook.
# Apply exactly the original filtering and physical-variable transformations.
data1 = np.genfromtxt(
    asset_path("combined.csv"), skip_header=1, delimiter=",", invalid_raise=False
)
data1 = data1[data1[:, 4] != 0]
for i in range(20):
    data1[:, i + 3] = np.sqrt(data1[:, i + 3]**2 - 2 * data1[:, 23])
data1[:, 1] = data1[:, 1] / np.sqrt(data1[:, -1])

X_tensor = torch.tensor(data1[:, :3], dtype=torch.float32)
Y_tensor = torch.tensor(data1[:, 3:23], dtype=torch.float32)
dataset = TensorDataset(X_tensor, Y_tensor)
n_train = int(0.8 * len(dataset))
# The original split happened immediately after torch.manual_seed(0), before
# model construction. A separate generator reproduces it without training.
_, test_set = random_split(
    dataset, [n_train, len(dataset) - n_train],
    generator=torch.Generator().manual_seed(0)
)
X_test = X_tensor[test_set.indices]
Y_test = Y_tensor[test_set.indices]
if len(X_test) < 25:
    raise ValueError("The cell-4 plot requires at least 25 test examples.")

import matplotlib.pyplot as plt
import numpy as np

# Predictions
model.eval()
with torch.no_grad():
    Y_pred = model(normX(X_test))
    Y_pred_denorm = denormy(Y_pred)

data_array = torch.cat([X_test, Y_test], dim=1).cpu().numpy()
model_array = torch.cat([X_test, Y_pred_denorm], dim=1).cpu().numpy()

vvec = np.array([
    0.000000, 0.020000, 0.080000, 0.180000, 0.320000,
    0.500000, 0.720000, 0.980000, 1.280000, 1.620000,
    2.000000, 2.420000, 2.880000, 3.380000, 3.920000,
    4.500000, 5.120000, 5.780000, 6.480000, 7.220000
])

plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 9,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

fig, ax = plt.subplots(figsize=(6.0, 4.2), dpi=300)

for i in range(5):

    idx = i + 20

    a, g, p = data_array[idx, :3]

    ax.plot(
        vvec,
        data_array[idx, 3:],
        linewidth=1.8,
        label=rf"Data ($\alpha={a:.2f},\ \gamma={g:.2f},\ \phi={p:.2f}$)"
    )

    ax.plot(
        vvec,
        model_array[idx, 3:],
        linestyle="None",
        marker="o",
        markersize=3,
        label=rf"NN"
    )

ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$v_{\rm cut}$')
ax.set_title('Neural-network prediction of cutoff profiles')

ax.grid(alpha=0.3)
ax.legend(frameon=False, ncol=1)

fig.tight_layout()
plt.show()
